# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library, following Croissant schema best practices.

### Dataset Source
The dataset's Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The metadata contains dataset-wide information, while records hold the tabular data defined by the record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show a concise summary of the dataset
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
List all record sets defined in the dataset along with their `@id`. For each, display the available fields and columns, with their respective `@id`s for precise referencing.

In [ ]:
# Record sets overview
print("Record sets and their fields:")
record_sets = list(dataset.record_sets())
for rs in record_sets:
    print(f"\nRecord Set: {rs.id}")
    for field in rs.fields:
        print(f"  Field: {field.id} (data_type={field.data_type})")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"    Column: {col.id} (source: {getattr(col, 'source', None)})")

## 3. Data Extraction
Load records of the main clinical dataset record set into a pandas DataFrame for processing.

**Note:** In this dataset, the main record set's `@id` will be displayed above (look for the typical table with ~77 cases). Replace the variable below with the appropriate value for `record_set_id`.

_For demonstration, we use the first available record set and field IDs. Adjust as needed to reference others._

In [ ]:
# Use the first record set found above (typically the tabular clinical data)
record_sets = list(dataset.record_sets())
main_record_set = record_sets[0]  # Replace index if required
record_set_id = main_record_set.id

print(f"Loading records from record set: {record_set_id}")

# Load all records for the chosen record set
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)

print("Available columns (field @id):")
print(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Carry out basic EDA: filtering, normalization, and grouping. All fields are referenced by their `@id`.

**Find a numeric field from the printed column list (field `@id`) above to use. We will filter and normalize it, then group by a categorical field.**

_Example fields for this dataset (replace as needed):_
- Numeric: `'http://senscience.ai/age'` (if available; or similar numeric field `@id`)
- Categorical: `'http://senscience.ai/sex'`, `'http://senscience.ai/msi_status'`, etc.

In [ ]:
# Example: Filter records with age > 50, normalize "age", and group by "msi_status"
# Replace the @ids below with those relevant to your dataset (as displayed in previous outputs)

numeric_field_id = None
group_field_id = None

# Try to guess a numeric field, e.g., one containing 'age', else use the first float/int field
for col in df.columns:
    if 'age' in col.lower() or 'interval' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # Fallback: use the first column with numeric dtype
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break

# Similarly, choose a categorical/group field
for col in df.columns:
    if 'msi' in col.lower() or 'sex' in col.lower() or 'location' in col.lower():
        group_field_id = col
        break

print(f"Numeric field selected: {numeric_field_id}")
print(f"Group-by field selected: {group_field_id}\n")

if numeric_field_id and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean() if np.isfinite(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} above mean ({threshold:.2f}) (n={len(filtered_df)}):")
    display(filtered_df[[numeric_field_id, group_field_id]].head()) if group_field_id else print(filtered_df.head())
    
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouped analysis
    if group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id]
            .mean()
            .reset_index()
            .rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
        )
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships using matplotlib or seaborn, referencing fields by their `@id`. For example, visualize the distribution of the numeric field or compare means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the histogram of the numeric field if available
if numeric_field_id and numeric_field_id in df:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Plot group means as barplot
if group_field_id and group_field_id in df and numeric_field_id in df:
    plt.figure(figsize=(7,4))
    grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
    sns.barplot(x=grouped.index, y=grouped.values)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR² dataset as defined by its Croissant schema via `mlcroissant`, explored its structure by `@id`, and performed basic tabular EDA, normalization, grouping, and visualization. 

- All references to data fields and record sets are made via their unique `@id`.
- Data processing steps can be further customized given more detailed knowledge of the clinical variables and analytical context.

_For more advanced analysis, consider exploring relationships among additional fields (by `@id`), applying statistical tests, or visualizing more complex interactions!_